# Checkpoint D2: Cross-team Testing
Chuan bi Skill package de gui cho nhom khac.

In [ ]:
import json
from pathlib import Path
import shutil

print('Imports OK: json, pathlib, shutil')

In [ ]:
# Define Skill package path
skill_package_path = Path('../../templates/skills/hr-policy-qa-skill').resolve()

print(f'Skill package path: {skill_package_path}')
print(f'Exists: {skill_package_path.exists()}')

if not skill_package_path.exists():
    print('ERROR: Skill package directory not found!')
    print('Please verify the path is correct.')

In [ ]:
# List all files in Skill package and verify completeness
all_files = []
for f in sorted(skill_package_path.rglob('*')):
    if f.is_file():
        rel_path = f.relative_to(skill_package_path)
        size = f.stat().st_size
        all_files.append({'path': str(rel_path), 'size': size, 'file': f})

# Expected files (8 required)
required_files = [
    'SKILL.md',
    'skill.json',
    'scripts/chunker.py',
    'scripts/evaluator.py',
    'scripts/retriever.py',
    'schemas/hr-response.schema.json',
    'kb/hr-policies/policy-allowance.md',
    'kb/hr-policies/policy-leave.md',
]

# Also check these optional but expected files
optional_files = [
    'kb/hr-policies/policy-seniority.md',
    'kb/hr-policies/policy-training.md',
    'kb/kb-inventory.md',
]

print(f'Total files found: {len(all_files)}')
print()
print(f'{"File Path":<45} {"Size":<10} {"Status"}')
print('=' * 70)

file_paths = {f['path'] for f in all_files}

for f in all_files:
    is_required = f['path'] in required_files
    is_optional = f['path'] in optional_files
    status = 'REQUIRED' if is_required else ('optional' if is_optional else 'extra')
    print(f'{f["path"]:<45} {f["size"]:>8} B  {status}')

# Check completeness
missing_required = [f for f in required_files if f not in file_paths]
print()
if missing_required:
    print(f'MISSING required files: {missing_required}')
else:
    print(f'All {len(required_files)} required files present.')

In [ ]:
# Validate SKILL.md has 6 sections
skill_md_path = skill_package_path / 'SKILL.md'

if skill_md_path.exists():
    content = skill_md_path.read_text(encoding='utf-8')
    
    # Count level-2 headings (## Section)
    import re
    sections = re.findall(r'^## (.+)$', content, re.MULTILINE)
    
    print(f'SKILL.md validation:')
    print(f'  File size: {len(content)} chars')
    print(f'  Sections found: {len(sections)}')
    print(f'  Required: 6 sections')
    print()
    print('  Sections:')
    for i, s in enumerate(sections, 1):
        print(f'    {i}. {s}')
    
    skill_md_valid = len(sections) >= 6
    print(f'\n  SKILL.md: {"PASS" if skill_md_valid else "FAIL"} ({len(sections)}/6 sections)')
else:
    skill_md_valid = False
    print('SKILL.md: NOT FOUND')

In [ ]:
# Validate skill.json has required fields
skill_json_path = skill_package_path / 'skill.json'

required_fields = ['name', 'version', 'triggers', 'permissions', 'quality_gates']

if skill_json_path.exists():
    with open(skill_json_path, 'r', encoding='utf-8') as f:
        skill_data = json.load(f)
    
    print('skill.json validation:')
    print(f'  Top-level keys: {list(skill_data.keys())}')
    print()
    
    skill_json_valid = True
    for field in required_fields:
        present = field in skill_data
        status = 'PASS' if present else 'FAIL'
        print(f'  {field}: {status}')
        if present:
            val = skill_data[field]
            if isinstance(val, (dict, list)):
                print(f'    Value: {json.dumps(val, ensure_ascii=False)[:100]}')
            else:
                print(f'    Value: {val}')
        if not present:
            skill_json_valid = False
    
    print(f'\n  skill.json: {"PASS" if skill_json_valid else "FAIL"}')
else:
    skill_json_valid = False
    print('skill.json: NOT FOUND')

In [ ]:
# Validate schema.json is valid JSON Schema
schema_path = skill_package_path / 'schemas' / 'hr-response.schema.json'

if schema_path.exists():
    with open(schema_path, 'r', encoding='utf-8') as f:
        schema_data = json.load(f)
    
    print('schema.json validation:')
    print(f'  File: {schema_path.name}')
    
    # Check JSON Schema required keywords
    schema_valid = True
    checks = {
        'has $schema': '$schema' in schema_data,
        'has type': 'type' in schema_data,
        'has properties or fields': 'properties' in schema_data or 'fields' in schema_data,
    }
    
    for check, passed in checks.items():
        print(f'  {check}: {"PASS" if passed else "WARN"}')
        if not passed:
            schema_valid = False
    
    # Print schema details
    print(f'\n  Schema content:')
    print(f'    $schema: {schema_data.get("$schema", "N/A")}')
    print(f'    type: {schema_data.get("type", "N/A")}')
    if 'properties' in schema_data:
        props = list(schema_data['properties'].keys())
        print(f'    properties: {props}')
    
    print(f'\n  schema.json: {"PASS" if schema_valid else "WARN"}')
else:
    schema_valid = False
    print('schema.json: NOT FOUND')

In [ ]:
# Validate scripts/ has 3 .py files
scripts_dir = skill_package_path / 'scripts'

if scripts_dir.exists():
    py_files = list(scripts_dir.glob('*.py'))
    expected_scripts = ['chunker.py', 'evaluator.py', 'retriever.py']
    
    print('scripts/ validation:')
    print(f'  Python files found: {len(py_files)}')
    print(f'  Expected: 3 (chunker.py, evaluator.py, retriever.py)')
    print()
    
    scripts_valid = True
    for expected in expected_scripts:
        exists = (scripts_dir / expected).exists()
        status = 'PASS' if exists else 'FAIL'
        print(f'  {expected}: {status}')
        if exists:
            size = (scripts_dir / expected).stat().st_size
            lines = len((scripts_dir / expected).read_text(encoding='utf-8').splitlines())
            print(f'    Size: {size} bytes, Lines: {lines}')
        if not exists:
            scripts_valid = False
    
    print(f'\n  scripts/: {"PASS" if scripts_valid else "FAIL"}')
else:
    scripts_valid = False
    print('scripts/: DIRECTORY NOT FOUND')

In [ ]:
# Cross-team readiness report
print('=' * 70)
print('CROSS-TEAM READINESS REPORT')
print('=' * 70)
print()

# File list summary
print(f'Files in package:     {len(all_files)}')
print(f'Required files:       {len(required_files) - len(missing_required)}/{len(required_files)}')
print()

# Validation results
validations = {
    'SKILL.md (6 sections)': skill_md_valid if 'skill_md_valid' in dir() else False,
    'skill.json (5 fields)': skill_json_valid if 'skill_json_valid' in dir() else False,
    'schema.json (valid)':   schema_valid if 'schema_valid' in dir() else False,
    'scripts/ (3 .py)':      scripts_valid if 'scripts_valid' in dir() else False,
    'Required files (8)':    len(missing_required) == 0 if 'missing_required' in dir() else False,
}

print('Validation Results:')
all_valid = True
for name, passed in validations.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'  {name:<30} {status}')
    if not passed:
        all_valid = False

print()
ready = 'YES' if all_valid else 'NO'
print(f'Ready to share: {ready}')

if not all_valid:
    print()
    print('Issues to fix:')
    for name, passed in validations.items():
        if not passed:
            print(f'  - {name}: needs attention')

In [ ]:
print('=' * 60)
print('CROSS-TEAM SHARING INSTRUCTIONS')
print('=' * 60)
print()
print('De chia se Skill package voi nhom khac:')
print()
print('1. Zip thu muc Skill package:')
print('   cd templates/skills/')
print('   zip -r hr-policy-qa-skill.zip hr-policy-qa-skill/')
print()
print('2. Gui file zip cho nhom khac (qua chat, email, hoac shared drive).')
print()
print('3. Nhom nhan lam theo huong dan:')
print('   a. Unzip: unzip hr-policy-qa-skill.zip')
print('   b. Cai dat dependencies: pip install -r requirements.txt')
print('   c. Chay evaluator: python scripts/evaluator.py')
print('   d. Bao cao ket qua: dien vao template bao cao')
print()
print('4. Nhom nhan can bao cao:')
print('   - So cau hoi pass/fail')
print('   - SLI metrics (accuracy, refusal rate, citation rate)')
print('   - Cau hoi fail kem ly do')
print('   - Gop y cai thien')
print()
print('5. Sau khi nhan feedback:')
print('   - Cap nhat Skill package')
print('   - Chay lai evaluator')
print('   - Gui ban cap nhat cho nhom review')
print()
print('CHECKPOINT D2 COMPLETE.')